In [1]:
import numpy as np
import xarray as xr
import uxarray as ux
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import Polygon, Point
import shapely

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'data/d-osp/gtsm'

In [3]:
def combine_mesh_datasets(dataset_paths):
    """
    Combine multiple unstructured mesh datasets into one, assuming time is identical for all datasets.

    Parameters:
        dataset_paths (list of str): List of file paths to NetCDF datasets.

    Returns:
        xarray.Dataset: Combined dataset with merged mesh topology, variables, and time coordinates.
    """
    datasets = [xr.open_dataset(path) for path in dataset_paths]

    # Initialize combined arrays for topology
    combined_nodes_x, combined_nodes_y = [], []
    combined_edges, combined_faces = [], []
    combined_face_bounds_x, combined_face_bounds_y = [], []
    combined_face_x, combined_face_y = [], []

    node_offset = 0
    data_vars = {}

    # Use time from the first dataset
    time_coord = datasets[0]['time'].values if 'time' in datasets[0].coords else None

    for ds in datasets:
        # Append node coordinates
        combined_nodes_x.append(ds['Mesh_node_x'].values)
        combined_nodes_y.append(ds['Mesh_node_y'].values)

        # Update edge connectivity with offset
        combined_edges.append(ds['Mesh_edge_nodes'].values + node_offset)

        # Update face connectivity with offset
        combined_faces.append(ds['Mesh_face_nodes'].values + node_offset)

        # Append face bounds
        combined_face_bounds_x.append(ds['Mesh_face_bounds_x'].values)
        combined_face_bounds_y.append(ds['Mesh_face_bounds_y'].values)

        combined_face_x.append(ds['Mesh_face_x'].values)
        combined_face_y.append(ds['Mesh_face_y'].values)


        # Merge all data variables except topology
        for var_name, var in ds.data_vars.items():
            if var_name in ['Mesh_node_x', 'Mesh_node_y', 'Mesh_edge_nodes',
                            'Mesh_face_nodes', 'Mesh_face_bounds_x', 'Mesh_face_bounds_y',
                            'Mesh_face_x', 'Mesh_face_y']:
                continue
            if var_name not in data_vars:
                data_vars[var_name] = []
            data_vars[var_name].append(var)

        node_offset += ds.dims['nMesh_node']

    # Concatenate topology arrays
    combined_nodes_x = np.concatenate(combined_nodes_x)
    combined_nodes_y = np.concatenate(combined_nodes_y)
    combined_edges = np.vstack(combined_edges)
    combined_faces = np.vstack(combined_faces)
    combined_face_bounds_x = np.vstack(combined_face_bounds_x)
    combined_face_bounds_y = np.vstack(combined_face_bounds_y)
    combined_face_x = np.concatenate(combined_face_x)
    combined_face_y = np.concatenate(combined_face_y)

    # Combine data variables along nMesh_face
    combined_data_vars = {}
    for var_name, var_list in data_vars.items():
        first_var = var_list[0]
        dims = first_var.dims

        if dims != ():
            if dims[0] == 'time':
                axis = 1
            else:
                axis = 0
            combined_data = np.concatenate([v.values for v in var_list], axis=axis)
            combined_data_vars[var_name] = (dims, combined_data)

        else:
            combined_data_vars[var_name] = ((), first_var.values)

    # No filtering: return full combined dataset
    coords = {
        'nMesh_node': np.arange(combined_nodes_x.shape[0]),
        'nMesh_edge': np.arange(combined_edges.shape[0]),
        'nMesh_face': np.arange(combined_faces.shape[0]),
        'Mesh_face_x': (('nMesh_face',), combined_face_x),
        'Mesh_face_y': (('nMesh_face',), combined_face_y)
    }
    if time_coord is not None:
        coords['time'] = time_coord

    combined_ds = xr.Dataset({
        'Mesh_node_x': (('nMesh_node',), combined_nodes_x),
        'Mesh_node_y': (('nMesh_node',), combined_nodes_y),
        'Mesh_edge_nodes': (('nMesh_edge', 'nMaxMesh_edge_nodes'), combined_edges),
        'Mesh_face_nodes': (('nMesh_face', 'nMaxMesh_face_nodes'), combined_faces),
        'Mesh_face_bounds_x': (('nMesh_face', 'nMaxMesh_face_bounds'), combined_face_bounds_x),
        'Mesh_face_bounds_y': (('nMesh_face', 'nMaxMesh_face_bounds'), combined_face_bounds_y),
        **combined_data_vars
    }, coords=coords)


    return combined_ds


In [4]:
def remap_array(arr, mapping):
    remapped = np.full_like(arr, np.nan)  # start with NaNs
    valid_mask = ~np.isnan(arr)
    # Convert only valid entries to int and map
    valid_values = arr[valid_mask].astype(int)
    remapped[valid_mask] = [mapping.get(x, -1) for x in valid_values]
    return remapped

In [5]:
def prepare_dataset(nc_files, aoi):
    combined_ds = combine_mesh_datasets(nc_files)
    
    face_pts = shapely.points(combined_ds['Mesh_face_x'].values,
                            combined_ds['Mesh_face_y'].values)
    node_pts = shapely.points(combined_ds['Mesh_node_x'].values,
                            combined_ds['Mesh_node_y'].values)
    edge_pts = shapely.points(combined_ds['Mesh_edge_x'].values,
                            combined_ds['Mesh_edge_y'].values)

    # Optional: prepared geometry can speed repeated predicates
    aoi_prep = shapely.prepare(aoi)

    face_mask = shapely.contains(aoi, face_pts)
    node_mask = shapely.contains(aoi, node_pts)
    edge_mask = shapely.contains(aoi, edge_pts)

    face_idx = np.flatnonzero(face_mask)
    node_idx = np.flatnonzero(node_mask)
    edge_idx = np.flatnonzero(edge_mask)

    ds_filtered = combined_ds.isel(
        nMesh_face=face_idx,
        nMesh_node=node_idx,
        nMesh_edge=edge_idx
    )

    face_nodes = ds_filtered.Mesh_face_nodes.values  # keep as float
    edge_nodes = ds_filtered.Mesh_edge_nodes.values  # keep as float

    # Suppose node_idx is the array of kept node indices after filtering
    # Example: node_idx = np.array([10, 11, 15, 20])  # old indices kept

    # Create mapping: old index -> new index
    mapping = {old: new for new, old in enumerate(node_idx)}

    # Apply remapping
    remapped_face_nodes = remap_array(face_nodes, mapping)
    remapped_edge_nodes = remap_array(edge_nodes, mapping)

    # Assign back to dataset
    ds_filtered["Mesh_face_nodes"] = (("nMesh_face", "nMaxMesh_face_nodes"), remapped_face_nodes)
    ds_filtered["Mesh_edge_nodes"] = (("nMesh_edge", "nMaxMesh_edge_nodes"), remapped_edge_nodes)

    return ds_filtered

In [9]:
relevant_datasets = ['01', '02', '09', '11', '13', '14']
nc_files = list(src_dir.glob("*gtsm_currents*.nc"))

aoi_gdf = gpd.read_file(src_dir / 'netherlands-domain.gpkg')
aoi = aoi_gdf['geometry'].iloc[0]

filtered_ds = prepare_dataset(nc_files=nc_files, aoi=aoi)
filtered_ds

/var/folders/1z/6xp0rndx2nqc7qx263rv42fh0000gn/T/ipykernel_58099/1341916160.py:54: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  node_offset += ds.dims['nMesh_node']
/Users/hemert/projects/d-osp/criticality-analysis/.venv/lib/python3.12/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


<xarray.Dataset> Size: 42MB
Dimensions:             (nMesh_node: 19477, nMesh_edge: 39076,
                         nMaxMesh_edge_nodes: 2, nMesh_face: 19601,
                         nMaxMesh_face_nodes: 4, nMaxMesh_face_bounds: 4,
                         time: 241)
Coordinates:
  * nMesh_node          (nMesh_node) int64 156kB 1788593 1788594 ... 4321400
  * nMesh_edge          (nMesh_edge) int64 313kB 3798867 3798868 ... 9012098
  * nMesh_face          (nMesh_face) int64 157kB 2016628 2016629 ... 4662763
  * time                (time) datetime64[ns] 2kB 2025-11-08T12:00:00 ... 202...
    Mesh_face_x         (nMesh_face) float64 157kB 7.332 7.332 ... 1.655 1.597
    Mesh_face_y         (nMesh_face) float64 157kB 53.31 53.3 ... 51.2 51.22
Dimensions without coordinates: nMaxMesh_edge_nodes, nMaxMesh_face_nodes,
                                nMaxMesh_face_bounds
Data variables:
    Mesh_node_x         (nMesh_node) float64 156kB 7.324 7.339 ... 2.329 2.241
    Mesh_node_y         (nMesh_node) float64 156kB 53.31 53.31 ... 51.04 51.02
    Mesh_edge_nodes     (nMesh_edge, nMaxMesh_edge_nodes) int32 313kB 1 ... 1...
    Mesh_face_nodes     (nMesh_face, nMaxMesh_face_nodes) float64 627kB 9.0 ....
    Mesh_face_bounds_x  (nMesh_face, nMaxMesh_face_bounds) float64 627kB 7.32...
    Mesh_face_bounds_y  (nMesh_face, nMaxMesh_face_bounds) float64 627kB 53.3...
    Mesh                int32 4B -2147483647
    Mesh_edge_x         (nMesh_edge) float64 313kB 7.332 7.339 ... 2.161 2.153
    Mesh_edge_y         (nMesh_edge) float64 313kB 53.31 53.31 ... 51.02 51.01
    crs                 int32 4B 0
    currents_u          (time, nMesh_face) float32 19MB 0.0 0.0 0.0 ... nan nan
    currents_v          (time, nMesh_face) float32 19MB 0.0 0.0 ... nan nan